# Sales Data — Cleaning

This notebook cleans the raw sales dataset using documented and reproducible rules. The raw Excel file is never modified. A cleaned workbook and a cleaning log are exported to `data/processed/cleaned_sales_data.xlsx`.

## 1. Import Libraries and Define Paths

In [1]:
from pathlib import Path

import pandas as pd

In [2]:
PROJECT_ROOT = Path.cwd()

if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DATA_PATH = PROJECT_ROOT / "data" / "raw" / "sales_data.xlsx"
PROCESSED_DATA_PATH = PROJECT_ROOT / "data" / "processed" / "cleaned_sales_data.xlsx"

print("Raw data:", RAW_DATA_PATH)
print("Processed data:", PROCESSED_DATA_PATH)
print("Raw dataset exists:", RAW_DATA_PATH.exists())

Raw data: c:\Users\Victus 16\sales-data-analysis\data\raw\sales_data.xlsx
Processed data: c:\Users\Victus 16\sales-data-analysis\data\processed\cleaned_sales_data.xlsx
Raw dataset exists: True


## 2. Load the Raw Dataset

A separate copy named `clean_df` is created so that the loaded raw DataFrame remains unchanged.

In [3]:
raw_df = pd.read_excel(
    RAW_DATA_PATH,
    sheet_name="Raw_Sales_Data",
)

clean_df = raw_df.copy()
cleaning_log = []

print("Raw rows:", len(raw_df))
print("Raw unique orders:", raw_df["Order_ID"].nunique())

Raw rows: 1295
Raw unique orders: 800


## 3. Remove Exact Duplicate Rows

Only completely identical rows are removed. Repeated `Order_ID` values are allowed because one order may contain multiple products.

In [4]:
duplicate_count = int(clean_df.duplicated().sum())
clean_df = clean_df.drop_duplicates().copy()

cleaning_log.append(
    {
        "Step": "Remove exact duplicates",
        "Rows_Affected": duplicate_count,
        "Decision": "Removed completely identical rows",
    }
)

print("Duplicates removed:", duplicate_count)
print("Rows remaining:", len(clean_df))

Duplicates removed: 8
Rows remaining: 1287


## 4. Standardize Text Values

Leading/trailing spaces are removed. City and payment values are mapped to their canonical forms.

In [5]:
PAYMENT_METHOD_MAP = {
    "credit card": "Credit Card",
    "debit card": "Debit Card",
    "paypal": "PayPal",
    "bank transfer": "Bank Transfer",
}

for column in ["Product", "Category", "City", "Payment_Method"]:
    clean_df[column] = clean_df[column].astype("string").str.strip()

non_standard_city_count = int(
    (clean_df["City"] != clean_df["City"].str.title()).fillna(False).sum()
)
clean_df["City"] = clean_df["City"].str.title()

clean_df["Payment_Method"] = (
    clean_df["Payment_Method"]
    .str.casefold()
    .map(PAYMENT_METHOD_MAP)
    .astype("string")
)

cleaning_log.append(
    {
        "Step": "Standardize text",
        "Rows_Affected": non_standard_city_count,
        "Decision": "Trimmed text and normalized city/payment spelling",
    }
)

print("Non-standard city capitalization corrected:", non_standard_city_count)

Non-standard city capitalization corrected: 8


## 5. Handle Missing Order-Level Values

City and payment method belong to the entire order. Missing values are first recovered from another line of the same `Order_ID`. Only values that cannot be recovered are labeled `Unknown`.

In [6]:
for column in ["City", "Payment_Method"]:
    missing_before = int(clean_df[column].isna().sum())

    clean_df[column] = clean_df.groupby("Order_ID")[column].transform(
        lambda values: values.ffill().bfill()
    )

    missing_after_recovery = int(clean_df[column].isna().sum())
    recovered = missing_before - missing_after_recovery

    clean_df[column] = clean_df[column].fillna("Unknown")

    cleaning_log.append(
        {
            "Step": f"Handle missing {column}",
            "Rows_Affected": missing_before,
            "Decision": (
                f"Recovered {recovered} from the same order; "
                f"labeled {missing_after_recovery} as Unknown"
            ),
        }
    )

    print(
        column,
        "- missing:", missing_before,
        "recovered:", recovered,
        "Unknown:", missing_after_recovery,
    )

City - missing: 10 recovered: 7 Unknown: 3
Payment_Method - missing: 8 recovered: 4 Unknown: 4


## 6. Convert and Validate Quantity

Quantity is line-level information and cannot be safely inferred. Rows with non-numeric, zero, or negative quantities are removed.

In [7]:
quantity_numeric = pd.to_numeric(
    clean_df["Quantity"],
    errors="coerce",
)

invalid_quantity_mask = (
    quantity_numeric.isna()
    | (quantity_numeric <= 0)
)
invalid_quantity_count = int(invalid_quantity_mask.sum())

clean_df = clean_df.loc[~invalid_quantity_mask].copy()
clean_df["Quantity"] = quantity_numeric.loc[
    ~invalid_quantity_mask
].astype("int64")

cleaning_log.append(
    {
        "Step": "Validate quantity",
        "Rows_Affected": invalid_quantity_count,
        "Decision": "Removed rows with unusable quantity values",
    }
)

print("Invalid quantity rows removed:", invalid_quantity_count)

Invalid quantity rows removed: 6


## 7. Repair Invalid Unit Prices

Invalid prices are replaced with the median valid price of the same product. Median is used because it is resistant to unusually high or low values.

In [8]:
invalid_price_mask = clean_df["Unit_Price_USD"] <= 0
invalid_price_count = int(invalid_price_mask.sum())

valid_product_medians = (
    clean_df.loc[~invalid_price_mask]
    .groupby("Product")["Unit_Price_USD"]
    .median()
)

clean_df.loc[invalid_price_mask, "Unit_Price_USD"] = (
    clean_df.loc[invalid_price_mask, "Product"]
    .map(valid_product_medians)
)

cleaning_log.append(
    {
        "Step": "Repair unit prices",
        "Rows_Affected": invalid_price_count,
        "Decision": "Replaced with median valid price for the same product",
    }
)

print("Invalid unit prices repaired:", invalid_price_count)

Invalid unit prices repaired: 5


## 8. Validate Discount Rates

A line-specific discount cannot be recovered reliably. Rows with discount rates outside 0–1 are removed rather than clipped to an arbitrary value.

In [9]:
invalid_discount_mask = ~clean_df["Discount_Rate"].between(0, 1)
invalid_discount_count = int(invalid_discount_mask.sum())

clean_df = clean_df.loc[~invalid_discount_mask].copy()

cleaning_log.append(
    {
        "Step": "Validate discount rates",
        "Rows_Affected": invalid_discount_count,
        "Decision": "Removed rows with discount rates outside 0-1",
    }
)

print("Invalid discount rows removed:", invalid_discount_count)

Invalid discount rows removed: 5


## 9. Parse and Validate Order Dates

Invalid or out-of-scope dates are first recovered from another line of the same order. Rows are removed only when recovery is impossible.

In [10]:
parsed_dates = pd.to_datetime(
    clean_df["Order_Date"],
    errors="coerce",
)

invalid_date_mask = (
    parsed_dates.isna()
    | (parsed_dates.dt.year != 2025)
)
invalid_date_count = int(invalid_date_mask.sum())
parsed_dates = parsed_dates.mask(invalid_date_mask)
clean_df["Order_Date"] = parsed_dates

clean_df["Order_Date"] = clean_df.groupby("Order_ID")["Order_Date"].transform(
    lambda values: values.ffill().bfill()
)

unrecoverable_date_count = int(clean_df["Order_Date"].isna().sum())
recovered_date_count = invalid_date_count - unrecoverable_date_count

clean_df = clean_df.dropna(subset=["Order_Date"]).copy()

cleaning_log.append(
    {
        "Step": "Validate order dates",
        "Rows_Affected": invalid_date_count,
        "Decision": (
            f"Recovered {recovered_date_count} from the same order; "
            f"removed {unrecoverable_date_count} unrecoverable rows"
        ),
    }
)

print("Invalid dates found:", invalid_date_count)
print("Dates recovered:", recovered_date_count)
print("Rows removed for invalid dates:", unrecoverable_date_count)

Invalid dates found: 5
Dates recovered: 4
Rows removed for invalid dates: 1


## 10. Correct Product Categories

Product categories are corrected using the approved product-to-category reference mapping.

In [11]:
PRODUCT_CATEGORY_MAP = {
    "Laptop": "Computers",
    "Monitor": "Computers",
    "Keyboard": "Accessories",
    "Wireless Mouse": "Accessories",
    "USB-C Hub": "Accessories",
    "Webcam": "Accessories",
    "Headphones": "Audio",
    "Bluetooth Speaker": "Audio",
    "External SSD": "Storage",
    "USB Flash Drive": "Storage",
    "Office Chair": "Furniture",
    "Desk Lamp": "Furniture",
}

expected_categories = clean_df["Product"].map(PRODUCT_CATEGORY_MAP)
category_mismatch_count = int(
    (clean_df["Category"] != expected_categories).sum()
)
clean_df["Category"] = expected_categories

cleaning_log.append(
    {
        "Step": "Correct product categories",
        "Rows_Affected": category_mismatch_count,
        "Decision": "Replaced with approved category for each product",
    }
)

print("Product/category mismatches corrected:", category_mismatch_count)

Product/category mismatches corrected: 5


## 11. Reset the Index and Review the Cleaning Log

In [12]:
clean_df = clean_df.reset_index(drop=True)
cleaning_log_df = pd.DataFrame(cleaning_log)
cleaning_log_df

,Step,Rows_Affected,Decision
0,Remove exact duplicates,8,Removed completely identical rows
1,Standardize text,8,Trimmed text and normalized city/payment spelling
2,Handle missing City,10,Recovered 7 from the same order; labeled 3 as ...
3,Handle missing Payment_Method,8,Recovered 4 from the same order; labeled 4 as ...
4,Validate quantity,6,Removed rows with unusable quantity values
5,Repair unit prices,5,Replaced with median valid price for the same ...
6,Validate discount rates,5,Removed rows with discount rates outside 0-1
7,Validate order dates,5,Recovered 4 from the same order; removed 1 unr...
8,Correct product categories,5,Replaced with approved category for each product


## 12. Final Validation

Assertions stop the notebook if a required quality rule is still violated.

In [13]:
assert clean_df.duplicated().sum() == 0
assert clean_df["Order_Line_ID"].duplicated().sum() == 0
assert clean_df.isna().sum().sum() == 0
assert (clean_df["Quantity"] > 0).all()
assert (clean_df["Unit_Price_USD"] > 0).all()
assert clean_df["Discount_Rate"].between(0, 1).all()
assert (clean_df["Order_Date"].dt.year == 2025).all()
assert clean_df["Category"].eq(
    clean_df["Product"].map(PRODUCT_CATEGORY_MAP)
).all()

print("All final validation checks passed.")
print("Clean rows:", len(clean_df))
print("Clean unique orders:", clean_df["Order_ID"].nunique())
print("Rows removed from raw data:", len(raw_df) - len(clean_df))

All final validation checks passed.
Clean rows: 1275
Clean unique orders: 797
Rows removed from raw data: 20


In [14]:
clean_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1275 entries, 0 to 1274
Data columns (total 11 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   Order_Line_ID   1275 non-null   str           
 1   Order_ID        1275 non-null   str           
 2   Order_Date      1275 non-null   datetime64[us]
 3   Customer_ID     1275 non-null   str           
 4   Product         1275 non-null   string        
 5   Category        1275 non-null   str           
 6   City            1275 non-null   string        
 7   Quantity        1275 non-null   int64         
 8   Unit_Price_USD  1275 non-null   float64       
 9   Discount_Rate   1275 non-null   float64       
 10  Payment_Method  1275 non-null   string        
dtypes: datetime64[us](1), float64(2), int64(1), str(4), string(3)
memory usage: 109.7 KB


## 13. Export the Cleaned Dataset

The workbook contains both the cleaned records and an audit log of the decisions applied.

In [15]:
PROCESSED_DATA_PATH.parent.mkdir(
    parents=True,
    exist_ok=True,
)

with pd.ExcelWriter(
    PROCESSED_DATA_PATH,
    engine="openpyxl",
) as writer:
    clean_df.to_excel(
        writer,
        sheet_name="Cleaned_Data",
        index=False,
    )

    cleaning_log_df.to_excel(
        writer,
        sheet_name="Cleaning_Log",
        index=False,
    )

print("Cleaned workbook exported successfully.")
print("Output:", PROCESSED_DATA_PATH)

Cleaned workbook exported successfully.
Output: c:\Users\Victus 16\sales-data-analysis\data\processed\cleaned_sales_data.xlsx


## Cleaning Result

- Raw rows: 1,295
- Final clean rows: 1,275
- Final unique orders: 797
- Exact duplicates remaining: 0
- Missing values remaining: 0
- Invalid quantities, prices, discounts, dates, and categories remaining: 0
- The original raw workbook was not modified.